# Part 3: Path Following

- **Path:** linear in joint space, `q_desired(λ) = (1 − λ)·q_start + λ·q_end`, with λ going from 0 to 1.
- **Trajectory:** `λ(t) = (Δλ/Δt)·t`. Every loop step commands the next point on the path, then waits `Δt`.
- **Measure:** read the actual joint angles back from the servos with the Dofbot software (`Arm_serial_servo_read`).
- **Error:** `q_desired(λ) − q(λ)` for each joint.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from Arm_Lib import Arm_Device

arm = Arm_Device()

# Start with the arm straight up
arm.Arm_serial_servo_write6(90, 90, 90, 90, 90, 90, 2000)
time.sleep(2.5)

## Make a joint space path
`N` steps means `Δλ = 1/N`. Angles are rounded to whole degrees, since that's what the servos take.

In [ ]:
def joint_path(q_start, q_end, N):
    lam = np.linspace(0, 1, N + 1)
    path = [(1 - l) * np.array(q_start) + l * np.array(q_end) for l in lam]
    return lam, np.round(path).astype(int)

## Follow a path
1. Move to the start of the path and wait.
2. For each point: command it (the servo gets `Δt` to get there), wait `Δt`, then read back all 5 joint angles.

Returns the actual angles and the time of each reading.

In [ ]:
def run_path(path, dt):
    # Go to the start of the path first
    arm.Arm_serial_servo_write6(*[int(a) for a in path[0]], 90, 2000)
    time.sleep(2.5)

    actual = []
    times = []
    t0 = time.time()
    for q in path:
        arm.Arm_serial_servo_write6(*[int(a) for a in q], 90, int(dt * 1000))
        time.sleep(dt)
        actual.append([arm.Arm_serial_servo_read(i) for i in range(1, 6)])
        times.append(time.time() - t0)
    return np.array(actual), np.array(times)

## Error summary and plot
Prints the RMS and max error for each joint, and plots the error along the path.

In [ ]:
colors = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']

def show_error(lam, path, actual, title):
    err = path - actual                      # q_desired - q_actual
    rms = np.sqrt(np.mean(err**2, axis=0))
    mx = np.max(np.abs(err), axis=0)

    print(title)
    print("RMS error per joint (deg):", np.round(rms, 2))
    print("Max error per joint (deg):", np.round(mx, 2))

    plt.figure(figsize=(7, 3.5))
    for j in range(5):
        plt.plot(lam, err[:, j], color=colors[j], linewidth=2, label=f"Joint {j+1}")
    plt.axhline(0, color='#999999', linewidth=1)
    plt.xlabel("λ")
    plt.ylabel("q_desired − q_actual (deg)")
    plt.title(title)
    plt.legend(frameon=False)
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()
    return err

## Run multiple paths
Three paths, each with `N = 50` steps (`Δλ = 0.02`) and `Δt = 0.1 s`, so each one takes about 5 s.

In [ ]:
paths = [
    ([90, 90, 90, 90, 90],    [45, 130, 60, 90, 90]),
    ([45, 130, 60, 90, 90],   [135, 130, 60, 130, 90]),
    ([135, 130, 60, 130, 90], [30, 140, 30, 150, 60]),
]

N = 50
dt = 0.1

for k, (q_start, q_end) in enumerate(paths):
    lam, path = joint_path(q_start, q_end, N)
    actual, times = run_path(path, dt)
    print(f"\nTotal time: {times[-1]:.2f} s")
    show_error(lam, path, actual, f"Path {k+1}: {q_start} to {q_end}")

# 6000 Level: Trajectory Exploration

**Same path, two trajectories.** Path 2 from above, with the same `Δλ = 0.02`, run at two different `Δt` values:
- **Slow:** `Δt = 0.2 s` (about 10 s total)
- **Fast:** `Δt = 0.03 s` (about 1.5 s total)

Changing `Δt` changes how fast λ moves (`λ(t) = (Δλ/Δt)·t`), so the arm travels the same path at a different speed.

**Prediction:** the servos need time to reach each commanded angle. In the slow run they mostly catch up before the next command, so the error should stay small. In the fast run they fall behind, so the error should be larger and mostly one sign (lagging behind the command). It should be biggest for the joints that move the most (joint 1 moves 90°, joint 4 moves 40°), and the arm should still be short of the end when the loop finishes.

In [ ]:
q_start, q_end = paths[1]
lam, path = joint_path(q_start, q_end, N)

actual_slow, times_slow = run_path(path, 0.2)
actual_fast, times_fast = run_path(path, 0.03)

err_slow = show_error(lam, path, actual_slow, "Slow trajectory (Δt = 0.2 s)")
err_fast = show_error(lam, path, actual_fast, "Fast trajectory (Δt = 0.03 s)")

## Compare the two trajectories
Total time, overall RMS error (all joints together), and the size of the error at the last point.

In [ ]:
for name, err, times in [("Slow", err_slow, times_slow), ("Fast", err_fast, times_fast)]:
    print(name)
    print("  Total time (s):", round(times[-1], 2))
    print("  Overall RMS error (deg):", round(np.sqrt(np.mean(err**2)), 2))
    print("  Error at end of path (deg):", err[-1])

plt.figure(figsize=(7, 3.5))
plt.plot(lam, np.linalg.norm(err_slow, axis=1), color=colors[0], linewidth=2, label="Slow (Δt = 0.2 s)")
plt.plot(lam, np.linalg.norm(err_fast, axis=1), color=colors[1], linewidth=2, label="Fast (Δt = 0.03 s)")
plt.xlabel("λ")
plt.ylabel("‖q_desired − q_actual‖ (deg)")
plt.title("Same path, two trajectories")
plt.legend(frameon=False)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()